# Medical Image Segmentation (U-Net & FCN) - Colab runner

**Before you start:** `Runtime > Change runtime type > T4 GPU`.  Then run the cells top to bottom.
This notebook only *calls* the scripts of the project, so everything you run here is exactly what is in the code files.

In [ ]:
# 1) GPU check
import torch
print("CUDA available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU - training will be very slow")

## 2) Get the project into Colab
Upload `medseg_project.zip` when the file picker opens.

In [ ]:
from google.colab import files
uploaded = files.upload()            # choose medseg_project.zip
!unzip -q -o medseg_project.zip
%cd medseg_project
!pip install -q kagglehub nibabel tabulate segmentation-models-pytorch

## 3) 1-minute smoke test with fake data (checks that everything is installed correctly)

In [ ]:
!python make_synthetic_data.py --out data/synthetic --layout folder
!python train.py --dataset folder --data-root data/synthetic --model unet --base-ch 8 --img-size 64 --epochs 3 --batch-size 8 --out runs/smoke
!python evaluate.py --run runs/smoke

## 4) Download the real dataset (Kaggle: LGG MRI Segmentation)
If Kaggle asks for credentials: create an API token at https://www.kaggle.com/settings and add it as a **Colab secret**
named `KAGGLE_API_TOKEN` (key icon in the left sidebar), then enable *Notebook access* for it.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
except Exception as e:
    print("No Colab secret found - kagglehub may ask you to log in:", e)

!python download_data.py --out data/lgg

In [ ]:
# Visualise data augmentations
!python show_augmentations.py --dataset lgg --data-root data/lgg --out figures/augmentations.png
from IPython.display import Image, display
display(Image("figures/augmentations.png"))

## 5) Train the models
`U-Net` is the main model. `FCN-32s` and `FCN-8s` give the comparison (and the skip-connection ablation).
Each run takes roughly 15-40 minutes on a free T4 GPU, depending on epochs.

In [ ]:
!python train.py --dataset lgg --data-root data/lgg --model unet   --epochs 50 --out runs/unet

In [ ]:
!python train.py --dataset lgg --data-root data/lgg --model fcn32s --epochs 40 --lr 1e-4 --out runs/fcn32s
!python train.py --dataset lgg --data-root data/lgg --model fcn8s  --epochs 40 --lr 1e-4 --out runs/fcn8s

In [ ]:
# Train without augmentation (ablation)
!python train.py --dataset lgg --data-root data/lgg --model unet --aug none --epochs 50 --out runs/unet_noaug

## 6) Evaluate on the held-out TEST patients and build the results table

In [ ]:
for r in ["unet", "fcn32s", "fcn8s", "unet_noaug"]:
    !python evaluate.py --run runs/{r}
!python compare_runs.py runs/unet runs/fcn32s runs/fcn8s runs/unet_noaug --out results

In [ ]:
from IPython.display import Image, display
display(Image("runs/unet/curves.png"))
display(Image("runs/unet/eval_test/qualitative_cases.png"))
display(Image("results/comparison.png"))

## 7) Save your results (Colab storage is deleted when the session ends)

In [ ]:
!zip -qr my_results.zip runs/*/config.json runs/*/split.json runs/*/history.csv runs/*/curves.png runs/*/best.pt runs/*/eval_test results figures
from google.colab import files
files.download("my_results.zip")